In [1]:
# Phase 11 - Advanced Detection Engine

import pandas as pd
import numpy as np
from pathlib import Path

# Identify project directories
project_path = Path.cwd()
data_path = project_path / "data"
processed_path = data_path / "processed"

print("Project path:", project_path)
print("Processed data path:", processed_path)
print("Directory exists:", processed_path.exists())

Project path: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets
Processed data path: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed
Directory exists: True


In [2]:
# Load cleaned datasets

baggage = pd.read_csv(processed_path / "baggage_clean.csv")

flight = pd.read_csv(processed_path / "flight_clean.csv")

baggage_assignment = pd.read_csv(
    processed_path / "baggage_assignment_clean.csv"
)

baggage_scan_event = pd.read_csv(
    processed_path / "baggage_scan_event_clean.csv"
)

print("All datasets loaded successfully!")

print("\nDataset shapes:")
print("Baggage:", baggage.shape)
print("Flight:", flight.shape)
print("Baggage Assignment:", baggage_assignment.shape)
print("Baggage Scan Event:", baggage_scan_event.shape)

All datasets loaded successfully!

Dataset shapes:
Baggage: (1000, 5)
Flight: (250, 6)
Baggage Assignment: (1500, 5)
Baggage Scan Event: (9000, 9)


In [3]:
# Convert timestamps into UTC datetime format

baggage_scan_event["event_time"] = pd.to_datetime(
    baggage_scan_event["event_time"],
    utc=True,
    errors="coerce"
)

baggage_scan_event["received_at"] = pd.to_datetime(
    baggage_scan_event["received_at"],
    utc=True,
    errors="coerce"
)

baggage_assignment["assigned_at"] = pd.to_datetime(
    baggage_assignment["assigned_at"],
    utc=True,
    errors="coerce"
)

flight["departure_time"] = pd.to_datetime(
    flight["departure_time"],
    utc=True,
    errors="coerce"
)

flight["arrival_time"] = pd.to_datetime(
    flight["arrival_time"],
    utc=True,
    errors="coerce"
)

print("Timestamp conversion completed!")

print("\nEvent time datatype:", baggage_scan_event["event_time"].dtype)
print("Received time datatype:", baggage_scan_event["received_at"].dtype)
print("Assignment time datatype:", baggage_assignment["assigned_at"].dtype)

Timestamp conversion completed!

Event time datatype: datetime64[us, UTC]
Received time datatype: datetime64[us, UTC]
Assignment time datatype: datetime64[us, UTC]


In [4]:
# Build expected flight segment reference

segment_reference = baggage_assignment.merge(
    flight[
        [
            "flight_id",
            "flight_number",
            "source",
            "destination",
            "departure_time",
            "arrival_time"
        ]
    ],
    on="flight_id",
    how="left",
    validate="many_to_one"
)

segment_reference = segment_reference.sort_values(
    ["bag_id", "segment_sequence"]
).reset_index(drop=True)

print("Flight segment reference created!")
print("Total segments:", len(segment_reference))
print("Unique baggage items:", segment_reference["bag_id"].nunique())

display(segment_reference.head(10))

Flight segment reference created!
Total segments: 1500
Unique baggage items: 1000


,assignment_id,bag_id,flight_id,segment_sequence,assigned_at,flight_number,source,destination,departure_time,arrival_time
0,ASG00001,BAG00001,FL0051,1,2026-11-01 10:00:00+00:00,6E5268,DEL,LHR,2026-11-01 15:00:00+00:00,2026-11-01 18:00:00+00:00
1,ASG00002,BAG00002,FL0102,1,2026-11-02 01:00:00+00:00,EK7887,DXB,JFK,2026-11-02 06:00:00+00:00,2026-11-02 12:00:00+00:00
2,ASG00003,BAG00003,FL0103,1,2026-11-03 01:00:00+00:00,LH2805,DXB,JFK,2026-11-03 06:00:00+00:00,2026-11-03 12:00:00+00:00
3,ASG00004,BAG00004,FL0044,1,2026-11-04 01:00:00+00:00,LH4110,MAA,DEL,2026-11-04 06:00:00+00:00,2026-11-04 09:00:00+00:00
4,ASG00005,BAG00005,FL0025,1,2026-11-05 01:00:00+00:00,6E811,MAA,SIN,2026-11-05 06:00:00+00:00,2026-11-05 09:00:00+00:00
5,ASG00006,BAG00006,FL0096,1,2026-11-06 10:00:00+00:00,EK9395,BOM,DXB,2026-11-06 15:00:00+00:00,2026-11-06 18:00:00+00:00
6,ASG00007,BAG00007,FL0087,1,2026-11-07 01:00:00+00:00,LH3358,FRA,LHR,2026-11-07 06:00:00+00:00,2026-11-07 12:00:00+00:00
7,ASG00008,BAG00008,FL0048,1,2026-11-08 01:00:00+00:00,EK5413,MAA,DEL,2026-11-08 06:00:00+00:00,2026-11-08 09:00:00+00:00
8,ASG00009,BAG00009,FL0099,1,2026-11-09 10:00:00+00:00,6E1233,BOM,DXB,2026-11-09 15:00:00+00:00,2026-11-09 18:00:00+00:00
9,ASG00010,BAG00010,FL0040,1,2026-11-10 10:00:00+00:00,QR3532,SIN,LHR,2026-11-10 15:00:00+00:00,2026-11-10 21:00:00+00:00


In [5]:
# Extract flight-related events

flight_event_types = [
    "SORTED",
    "LOADED",
    "ARRIVED",
    "UNLOADED"
]

actual_flight_events = baggage_scan_event[
    baggage_scan_event["event_type"].isin(flight_event_types)
].copy()

print("Flight-related scan events:", len(actual_flight_events))

# Match actual events with expected flight segments

segment_events = actual_flight_events.merge(
    segment_reference[
        [
            "bag_id",
            "segment_sequence",
            "flight_number",
            "source",
            "destination"
        ]
    ],
    on=["bag_id", "flight_number"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("\nEvent matching results:")
display(segment_events["_merge"].value_counts())

print("\nSample matched events:")
display(
    segment_events[
        [
            "bag_id",
            "event_type",
            "airport_code",
            "flight_number",
            "segment_sequence",
            "source",
            "destination"
        ]
    ].head(10)
)

Flight-related scan events: 5500

Event matching results:


_merge
both          5500
left_only        0
right_only       0
Name: count, dtype: int64


Sample matched events:


,bag_id,event_type,airport_code,flight_number,segment_sequence,source,destination
0,BAG00001,SORTED,DEL,6E5268,1,DEL,LHR
1,BAG00001,LOADED,DEL,6E5268,1,DEL,LHR
2,BAG00001,ARRIVED,LHR,6E5268,1,DEL,LHR
3,BAG00001,UNLOADED,LHR,6E5268,1,DEL,LHR
4,BAG00002,SORTED,DXB,EK7887,1,DXB,JFK
5,BAG00002,LOADED,DXB,EK7887,1,DXB,JFK
6,BAG00002,ARRIVED,JFK,EK7887,1,DXB,JFK
7,BAG00002,UNLOADED,JFK,EK7887,1,DXB,JFK
8,BAG00003,SORTED,DXB,LH2805,1,DXB,JFK
9,BAG00003,LOADED,DXB,LH2805,1,DXB,JFK


In [6]:
# Validate flight event airport association

segment_events["expected_airport"] = np.where(
    segment_events["event_type"].isin(["SORTED", "LOADED"]),
    segment_events["source"],
    segment_events["destination"]
)

segment_events["airport_matches"] = (
    segment_events["airport_code"] ==
    segment_events["expected_airport"]
)

print("Total flight events:", len(segment_events))

print("\nAirport validation results:")
display(segment_events["airport_matches"].value_counts())

print("\nEvents with airport mismatches:")

airport_mismatches = segment_events[
    ~segment_events["airport_matches"]
]

display(
    airport_mismatches[
        [
            "bag_id",
            "event_type",
            "flight_number",
            "airport_code",
            "expected_airport"
        ]
    ].head(20)
)

Total flight events: 5500

Airport validation results:


airport_matches
True    5500
Name: count, dtype: int64


Events with airport mismatches:


,bag_id,event_type,flight_number,airport_code,expected_airport


In [7]:
# Validate event sequence within each flight segment

expected_sequence = {
    "SORTED": 1,
    "LOADED": 2,
    "ARRIVED": 3,
    "UNLOADED": 4
}

segment_events["expected_event_order"] = (
    segment_events["event_type"].map(expected_sequence)
)

segment_events = segment_events.sort_values(
    ["bag_id", "segment_sequence", "event_time"]
)

segment_events["previous_event_order"] = (
    segment_events.groupby(
        ["bag_id", "segment_sequence"]
    )["expected_event_order"].shift(1)
)

segment_events["sequence_valid"] = (
    segment_events["expected_event_order"] >=
    segment_events["previous_event_order"]
)

sequence_results = segment_events.groupby(
    ["bag_id", "segment_sequence"]
)["sequence_valid"].all()

print("Flight segments checked:", len(sequence_results))

print("\nSegment sequence results:")
display(sequence_results.value_counts())

print(
    "\nSegments with sequence issues:",
    (~sequence_results).sum()
)

Flight segments checked: 1500

Segment sequence results:


sequence_valid
False    1500
Name: count, dtype: int64


Segments with sequence issues: 1500


In [8]:
# Correct segment-level event sequence validation

segment_events["sequence_valid"] = (
    segment_events["previous_event_order"].isna()
    |
    (
        segment_events["expected_event_order"]
        >= segment_events["previous_event_order"]
    )
)

sequence_results = segment_events.groupby(
    ["bag_id", "segment_sequence"]
)["sequence_valid"].all()

print("Flight segments checked:", len(sequence_results))

print("\nCorrected segment sequence results:")
display(sequence_results.value_counts())

print(
    "\nSegments with sequence issues:",
    (~sequence_results).sum()
)

Flight segments checked: 1500

Corrected segment sequence results:


sequence_valid
True    1500
Name: count, dtype: int64


Segments with sequence issues: 0


In [9]:
# Validate required events within each flight segment

required_segment_events = {
    "SORTED",
    "LOADED",
    "ARRIVED",
    "UNLOADED"
}

segment_completeness = (
    segment_events
    .groupby(["bag_id", "segment_sequence"])["event_type"]
    .apply(lambda events: required_segment_events.issubset(set(events)))
)

print("Total flight segments:", len(segment_completeness))

print("\nSegment completeness results:")
display(segment_completeness.value_counts())

print(
    "\nSegments missing required events:",
    (~segment_completeness).sum()
)

Total flight segments: 1500

Segment completeness results:


event_type
True     1000
False     500
Name: count, dtype: int64


Segments missing required events: 500


In [10]:
# Identify missing events for each flight segment

segment_event_sets = (
    segment_events
    .groupby(["bag_id", "segment_sequence"])["event_type"]
    .apply(set)
)

missing_event_records = []

for (bag_id, segment_sequence), actual_events in segment_event_sets.items():

    missing_events = required_segment_events - actual_events

    for event in missing_events:
        missing_event_records.append({
            "bag_id": bag_id,
            "segment_sequence": segment_sequence,
            "missing_event": event
        })

missing_events_df = pd.DataFrame(missing_event_records)

print("Total missing event records:", len(missing_events_df))

print("\nMissing event distribution:")
display(missing_events_df["missing_event"].value_counts())

print("\nMissing events by segment:")
display(
    missing_events_df.groupby(
        ["segment_sequence", "missing_event"]
    ).size().reset_index(name="missing_count")
)

print("\nSample missing event records:")
display(missing_events_df.head(10))

Total missing event records: 500

Missing event distribution:


missing_event
SORTED    500
Name: count, dtype: int64


Missing events by segment:


,segment_sequence,missing_event,missing_count
0,2,SORTED,500



Sample missing event records:


,bag_id,segment_sequence,missing_event
0,BAG00501,2,SORTED
1,BAG00502,2,SORTED
2,BAG00503,2,SORTED
3,BAG00504,2,SORTED
4,BAG00505,2,SORTED
5,BAG00506,2,SORTED
6,BAG00507,2,SORTED
7,BAG00508,2,SORTED
8,BAG00509,2,SORTED
9,BAG00510,2,SORTED


In [11]:
# Save segment completeness findings

output_path = processed_path / "phase11_missing_segment_events.csv"

missing_events_df.to_csv(output_path, index=False)

print("Missing segment event report saved!")
print("File:", output_path)
print("Total records:", len(missing_events_df))

Missing segment event report saved!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase11_missing_segment_events.csv
Total records: 500


In [12]:
# Prepare handoff-related events

handoff_events = baggage_scan_event[
    baggage_scan_event["event_type"].isin(
        ["UNLOADED", "TRANSFER", "LOADED"]
    )
].copy()

handoff_events = handoff_events.sort_values(
    ["bag_id", "event_time"]
)

print("Handoff-related events:", len(handoff_events))
print("Unique baggage items:", handoff_events["bag_id"].nunique())

display(
    handoff_events[
        ["bag_id", "event_type", "airport_code",
         "flight_number", "event_time"]
    ].head(15)
)

Handoff-related events: 3500
Unique baggage items: 1000


,bag_id,event_type,airport_code,flight_number,event_time
3,BAG00001,LOADED,DEL,6E5268,2026-11-01 14:15:00+00:00
5,BAG00001,UNLOADED,LHR,6E5268,2026-11-01 18:15:00+00:00
10,BAG00002,LOADED,DXB,EK7887,2026-11-02 05:15:00+00:00
12,BAG00002,UNLOADED,JFK,EK7887,2026-11-02 12:15:00+00:00
17,BAG00003,LOADED,DXB,LH2805,2026-11-03 05:15:00+00:00
19,BAG00003,UNLOADED,JFK,LH2805,2026-11-03 12:15:00+00:00
24,BAG00004,LOADED,MAA,LH4110,2026-11-04 05:15:00+00:00
26,BAG00004,UNLOADED,DEL,LH4110,2026-11-04 09:15:00+00:00
31,BAG00005,LOADED,MAA,6E811,2026-11-05 05:15:00+00:00
33,BAG00005,UNLOADED,SIN,6E811,2026-11-05 09:15:00+00:00


In [13]:
# Identify connecting baggage items

connecting_bag_ids = (
    baggage_assignment.groupby("bag_id")
    .size()
    .loc[lambda counts: counts == 2]
    .index
)

# Extract handoff events for connecting baggage

connecting_handoff_events = handoff_events[
    handoff_events["bag_id"].isin(connecting_bag_ids)
].copy()

print("Connecting baggage items:", len(connecting_bag_ids))
print("Connecting handoff events:", len(connecting_handoff_events))

# Inspect one normal and one inconsistent journey

for bag_id in ["BAG00502", "BAG00501"]:
    print(f"\nHandoff events for {bag_id}")

    display(
        connecting_handoff_events[
            connecting_handoff_events["bag_id"] == bag_id
        ][
            [
                "bag_id",
                "event_type",
                "airport_code",
                "flight_number",
                "event_time"
            ]
        ]
    )

Connecting baggage items: 500
Connecting handoff events: 2500

Handoff events for BAG00502


,bag_id,event_type,airport_code,flight_number,event_time
3514,BAG00502,LOADED,MAA,AI4606,2026-11-02 05:15:00+00:00
3516,BAG00502,UNLOADED,DXB,AI4606,2026-11-02 09:15:00+00:00
3517,BAG00502,TRANSFER,DXB,NaN,2026-11-02 09:45:00+00:00
3518,BAG00502,LOADED,DXB,EK9028,2026-11-02 14:15:00+00:00
3520,BAG00502,UNLOADED,LHR,EK9028,2026-11-02 21:15:00+00:00



Handoff events for BAG00501


,bag_id,event_type,airport_code,flight_number,event_time
3507,BAG00501,LOADED,DXB,6E2160,2026-11-01 05:15:00+00:00
3509,BAG00501,UNLOADED,JFK,6E2160,2026-11-01 12:15:00+00:00
3503,BAG00501,LOADED,BOM,AI9913,2026-11-01 14:15:00+00:00
3505,BAG00501,UNLOADED,DXB,AI9913,2026-11-01 18:15:00+00:00
3506,BAG00501,TRANSFER,DXB,NaN,2026-11-01 18:45:00+00:00


In [15]:
# Extract first-flight and second-flight details

first_flight = segment_reference[
    segment_reference["segment_sequence"] == 1
][["bag_id", "destination"]].rename(
    columns={"destination": "connecting_airport"}
)

second_flight = segment_reference[
    segment_reference["segment_sequence"] == 2
][["bag_id", "flight_number"]].rename(
    columns={"flight_number": "second_flight_number"}
)

# Extract first-flight UNLOADED events

first_unloaded = connecting_handoff_events[
    connecting_handoff_events["event_type"] == "UNLOADED"
].merge(
    first_flight,
    on="bag_id",
    how="inner"
)

first_unloaded = first_unloaded[
    first_unloaded["airport_code"] ==
    first_unloaded["connecting_airport"]
][["bag_id", "event_time"]].rename(
    columns={"event_time": "first_unloaded_time"}
)

# Extract TRANSFER events

transfer_events = connecting_handoff_events[
    connecting_handoff_events["event_type"] == "TRANSFER"
][["bag_id", "airport_code", "event_time"]].rename(
    columns={
        "airport_code": "transfer_airport",
        "event_time": "transfer_time"
    }
)

# Extract second-flight LOADED events

second_loaded = connecting_handoff_events[
    connecting_handoff_events["event_type"] == "LOADED"
].merge(
    second_flight,
    left_on=["bag_id", "flight_number"],
    right_on=["bag_id", "second_flight_number"],
    how="inner",
    validate="many_to_one"
)

second_loaded = second_loaded[
    ["bag_id", "airport_code", "event_time"]
].rename(
    columns={
        "airport_code": "second_loaded_airport",
        "event_time": "second_loaded_time"
    }
)

print("First unload records:", len(first_unloaded))
print("Transfer records:", len(transfer_events))
print("Second load records:", len(second_loaded))

First unload records: 500
Transfer records: 500
Second load records: 500


In [16]:
# Combine handoff events into one dataset

handoff_summary = (
    first_unloaded
    .merge(transfer_events, on="bag_id", how="inner")
    .merge(second_loaded, on="bag_id", how="inner")
)

# Calculate time gaps in minutes

handoff_summary["unload_to_transfer_minutes"] = (
    handoff_summary["transfer_time"] -
    handoff_summary["first_unloaded_time"]
).dt.total_seconds() / 60

handoff_summary["transfer_to_load_minutes"] = (
    handoff_summary["second_loaded_time"] -
    handoff_summary["transfer_time"]
).dt.total_seconds() / 60

print("Total connecting journeys:", len(handoff_summary))

display(
    handoff_summary[
        [
            "bag_id",
            "first_unloaded_time",
            "transfer_time",
            "second_loaded_time",
            "unload_to_transfer_minutes",
            "transfer_to_load_minutes"
        ]
    ].head(10)
)

Total connecting journeys: 500


,bag_id,first_unloaded_time,transfer_time,second_loaded_time,unload_to_transfer_minutes,transfer_to_load_minutes
0,BAG00501,2026-11-01 18:15:00+00:00,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,30.0,-810.0
1,BAG00502,2026-11-02 09:15:00+00:00,2026-11-02 09:45:00+00:00,2026-11-02 14:15:00+00:00,30.0,270.0
2,BAG00503,2026-11-03 09:15:00+00:00,2026-11-03 09:45:00+00:00,2026-11-03 14:15:00+00:00,30.0,270.0
3,BAG00504,2026-11-04 09:15:00+00:00,2026-11-04 09:45:00+00:00,2026-11-04 14:15:00+00:00,30.0,270.0
4,BAG00505,2026-11-05 09:15:00+00:00,2026-11-05 09:45:00+00:00,2026-11-05 14:15:00+00:00,30.0,270.0
5,BAG00506,2026-11-06 18:15:00+00:00,2026-11-06 18:45:00+00:00,2026-11-06 05:15:00+00:00,30.0,-810.0
6,BAG00507,2026-11-07 09:15:00+00:00,2026-11-07 09:45:00+00:00,2026-11-07 14:15:00+00:00,30.0,270.0
7,BAG00508,2026-11-08 09:15:00+00:00,2026-11-08 09:45:00+00:00,2026-11-08 14:15:00+00:00,30.0,270.0
8,BAG00509,2026-11-09 18:15:00+00:00,2026-11-09 18:45:00+00:00,2026-11-09 05:15:00+00:00,30.0,-810.0
9,BAG00510,2026-11-10 18:15:00+00:00,2026-11-10 18:45:00+00:00,2026-11-10 05:15:00+00:00,30.0,-810.0


In [17]:
# Validate handoff chronology

handoff_summary["unload_before_transfer"] = (
    handoff_summary["unload_to_transfer_minutes"] >= 0
)

handoff_summary["transfer_before_load"] = (
    handoff_summary["transfer_to_load_minutes"] >= 0
)

handoff_summary["handoff_chronology_valid"] = (
    handoff_summary["unload_before_transfer"] &
    handoff_summary["transfer_before_load"]
)

print("Total connecting journeys:", len(handoff_summary))

print("\nHandoff chronology results:")
display(
    handoff_summary["handoff_chronology_valid"].value_counts()
)

print(
    "\nInvalid handoff journeys:",
    (~handoff_summary["handoff_chronology_valid"]).sum()
)

print("\nHandoff duration statistics:")
display(
    handoff_summary[
        [
            "unload_to_transfer_minutes",
            "transfer_to_load_minutes"
        ]
    ].describe()
)

Total connecting journeys: 500

Handoff chronology results:


handoff_chronology_valid
True     315
False    185
Name: count, dtype: int64


Invalid handoff journeys: 185

Handoff duration statistics:


,unload_to_transfer_minutes,transfer_to_load_minutes
count,500.0,500.000000
mean,30.0,-129.600000
std,0.0,521.950865
min,30.0,-810.000000
25%,30.0,-810.000000
50%,30.0,270.000000
75%,30.0,270.000000
max,30.0,270.000000


In [18]:
# Validate connecting airport consistency

airport_validation = (
    first_flight
    .merge(
        second_flight.merge(
            segment_reference[
                segment_reference["segment_sequence"] == 2
            ][["bag_id", "source"]],
            on="bag_id",
            how="inner"
        ),
        on="bag_id",
        how="inner"
    )
    .merge(
        transfer_events[["bag_id", "transfer_airport"]],
        on="bag_id",
        how="inner"
    )
)

airport_validation["first_flight_matches"] = (
    airport_validation["connecting_airport"] ==
    airport_validation["source"]
)

airport_validation["transfer_matches"] = (
    airport_validation["connecting_airport"] ==
    airport_validation["transfer_airport"]
)

airport_validation["airport_consistent"] = (
    airport_validation["first_flight_matches"] &
    airport_validation["transfer_matches"]
)

print("Connecting journeys checked:", len(airport_validation))

print("\nAirport consistency results:")
display(airport_validation["airport_consistent"].value_counts())

Connecting journeys checked: 500

Airport consistency results:


airport_consistent
True    500
Name: count, dtype: int64

In [19]:
# Save advanced handoff investigation report

handoff_report = handoff_summary.merge(
    airport_validation[
        [
            "bag_id",
            "connecting_airport",
            "transfer_airport",
            "airport_consistent"
        ]
    ],
    on="bag_id",
    how="left"
)

handoff_report["handoff_issue"] = np.where(
    handoff_report["handoff_chronology_valid"],
    "NO_TIMING_ISSUE",
    "TRANSFER_LOADING_TIME_INCONSISTENCY"
)

handoff_output_path = (
    processed_path / "phase11_handoff_investigation.csv"
)

handoff_report.to_csv(handoff_output_path, index=False)

print("Handoff investigation report saved!")
print("File:", handoff_output_path)
print("Total records:", len(handoff_report))

print("\nIssue distribution:")
display(handoff_report["handoff_issue"].value_counts())

Handoff investigation report saved!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase11_handoff_investigation.csv
Total records: 500

Issue distribution:


handoff_issue
NO_TIMING_ISSUE                        315
TRANSFER_LOADING_TIME_INCONSISTENCY    185
Name: count, dtype: int64

In [20]:
# Prepare chronological scan events for duration analysis

duration_events = baggage_scan_event.copy()

duration_events = duration_events.sort_values(
    ["bag_id", "event_time"]
).reset_index(drop=True)

duration_events["previous_event"] = (
    duration_events.groupby("bag_id")["event_type"].shift(1)
)

duration_events["previous_event_time"] = (
    duration_events.groupby("bag_id")["event_time"].shift(1)
)

duration_events["duration_minutes"] = (
    duration_events["event_time"] -
    duration_events["previous_event_time"]
).dt.total_seconds() / 60

print("Total scan events:", len(duration_events))
print("Events with previous event:", duration_events["previous_event"].notna().sum())

display(
    duration_events[
        [
            "bag_id",
            "previous_event",
            "event_type",
            "previous_event_time",
            "event_time",
            "duration_minutes"
        ]
    ].head(15)
)

Total scan events: 9000
Events with previous event: 8000


,bag_id,previous_event,event_type,previous_event_time,event_time,duration_minutes
0,BAG00001,NaN,CHECK_IN,NaT,2026-11-01 11:00:00+00:00,NaN
1,BAG00001,CHECK_IN,SECURITY_SCAN,2026-11-01 11:00:00+00:00,2026-11-01 12:00:00+00:00,60.0
2,BAG00001,SECURITY_SCAN,SORTED,2026-11-01 12:00:00+00:00,2026-11-01 13:00:00+00:00,60.0
3,BAG00001,SORTED,LOADED,2026-11-01 13:00:00+00:00,2026-11-01 14:15:00+00:00,75.0
4,BAG00001,LOADED,ARRIVED,2026-11-01 14:15:00+00:00,2026-11-01 18:00:00+00:00,225.0
5,BAG00001,ARRIVED,UNLOADED,2026-11-01 18:00:00+00:00,2026-11-01 18:15:00+00:00,15.0
6,BAG00001,UNLOADED,DELIVERED,2026-11-01 18:15:00+00:00,2026-11-01 19:00:00+00:00,45.0
7,BAG00002,NaN,CHECK_IN,NaT,2026-11-02 02:00:00+00:00,NaN
8,BAG00002,CHECK_IN,SECURITY_SCAN,2026-11-02 02:00:00+00:00,2026-11-02 03:00:00+00:00,60.0
9,BAG00002,SECURITY_SCAN,SORTED,2026-11-02 03:00:00+00:00,2026-11-02 04:00:00+00:00,60.0


In [21]:
# Summarize duration for each consecutive event transition

duration_summary = (
    duration_events
    .dropna(subset=["duration_minutes"])
    .groupby(["previous_event", "event_type"])["duration_minutes"]
    .agg(
        event_count="count",
        average_minutes="mean",
        median_minutes="median",
        minimum_minutes="min",
        maximum_minutes="max"
    )
    .reset_index()
    .sort_values(["previous_event", "event_type"])
)

display(duration_summary.round(2))

,previous_event,event_type,event_count,average_minutes,median_minutes,minimum_minutes,maximum_minutes
0,ARRIVED,UNLOADED,1500,15.00,15.0,15.0,15.0
1,CHECK_IN,SECURITY_SCAN,1000,60.00,60.0,60.0,60.0
2,DELIVERED,LOADED,185,75.00,75.0,75.0,75.0
3,LOADED,ARRIVED,1315,287.56,225.0,225.0,405.0
4,LOADED,CHECK_IN,185,345.00,345.0,345.0,345.0
5,SECURITY_SCAN,ARRIVED,185,0.00,0.0,0.0,0.0
6,SECURITY_SCAN,SORTED,815,60.00,60.0,60.0,60.0
7,SORTED,DELIVERED,185,0.00,0.0,0.0,0.0
8,SORTED,LOADED,815,75.00,75.0,75.0,75.0
9,TRANSFER,LOADED,315,270.00,270.0,270.0,270.0


In [22]:
# Identify transitions with variable durations

transition_variability = (
    duration_events
    .dropna(subset=["duration_minutes"])
    .groupby(["previous_event", "event_type"])
    .agg(
        event_count=("duration_minutes", "count"),
        unique_durations=("duration_minutes", "nunique"),
        minimum_minutes=("duration_minutes", "min"),
        maximum_minutes=("duration_minutes", "max"),
        standard_deviation=("duration_minutes", "std")
    )
    .reset_index()
)

transition_variability["duration_varies"] = (
    transition_variability["unique_durations"] > 1
)

display(transition_variability.round(2))

,previous_event,event_type,event_count,unique_durations,minimum_minutes,maximum_minutes,standard_deviation,duration_varies
0,ARRIVED,UNLOADED,1500,1,15.0,15.0,0.00,False
1,CHECK_IN,SECURITY_SCAN,1000,1,60.0,60.0,0.00,False
2,DELIVERED,LOADED,185,1,75.0,75.0,0.00,False
3,LOADED,ARRIVED,1315,2,225.0,405.0,85.75,True
4,LOADED,CHECK_IN,185,1,345.0,345.0,0.00,False
5,SECURITY_SCAN,ARRIVED,185,1,0.0,0.0,0.00,False
6,SECURITY_SCAN,SORTED,815,1,60.0,60.0,0.00,False
7,SORTED,DELIVERED,185,1,0.0,0.0,0.00,False
8,SORTED,LOADED,815,1,75.0,75.0,0.00,False
9,TRANSFER,LOADED,315,1,270.0,270.0,0.00,False


In [23]:
# Prepare actual flight duration records

loaded_events = baggage_scan_event[
    baggage_scan_event["event_type"] == "LOADED"
][["bag_id", "flight_number", "event_time"]].rename(
    columns={"event_time": "actual_loaded_time"}
)

arrived_events = baggage_scan_event[
    baggage_scan_event["event_type"] == "ARRIVED"
][["bag_id", "flight_number", "event_time"]].rename(
    columns={"event_time": "actual_arrived_time"}
)

actual_flight_durations = loaded_events.merge(
    arrived_events,
    on=["bag_id", "flight_number"],
    how="inner",
    validate="one_to_one"
)

actual_flight_durations["actual_duration_minutes"] = (
    actual_flight_durations["actual_arrived_time"]
    - actual_flight_durations["actual_loaded_time"]
).dt.total_seconds() / 60

print("Matched actual flight journeys:", len(actual_flight_durations))

display(actual_flight_durations.head(10))

Matched actual flight journeys: 1500


,bag_id,flight_number,actual_loaded_time,actual_arrived_time,actual_duration_minutes
0,BAG00001,6E5268,2026-11-01 14:15:00+00:00,2026-11-01 18:00:00+00:00,225.0
1,BAG00002,EK7887,2026-11-02 05:15:00+00:00,2026-11-02 12:00:00+00:00,405.0
2,BAG00003,LH2805,2026-11-03 05:15:00+00:00,2026-11-03 12:00:00+00:00,405.0
3,BAG00004,LH4110,2026-11-04 05:15:00+00:00,2026-11-04 09:00:00+00:00,225.0
4,BAG00005,6E811,2026-11-05 05:15:00+00:00,2026-11-05 09:00:00+00:00,225.0
5,BAG00006,EK9395,2026-11-06 14:15:00+00:00,2026-11-06 18:00:00+00:00,225.0
6,BAG00007,LH3358,2026-11-07 05:15:00+00:00,2026-11-07 12:00:00+00:00,405.0
7,BAG00008,EK5413,2026-11-08 05:15:00+00:00,2026-11-08 09:00:00+00:00,225.0
8,BAG00009,6E1233,2026-11-09 14:15:00+00:00,2026-11-09 18:00:00+00:00,225.0
9,BAG00010,QR3532,2026-11-10 14:15:00+00:00,2026-11-10 21:00:00+00:00,405.0


In [24]:
# Join actual flight durations with scheduled flight information

flight_schedule = flight[
    [
        "flight_number",
        "source",
        "destination",
        "departure_time",
        "arrival_time"
    ]
].copy()

flight_schedule["scheduled_duration_minutes"] = (
    flight_schedule["arrival_time"]
    - flight_schedule["departure_time"]
).dt.total_seconds() / 60

flight_duration_analysis = actual_flight_durations.merge(
    flight_schedule,
    on="flight_number",
    how="left",
    validate="many_to_one"
)

flight_duration_analysis["duration_difference_minutes"] = (
    flight_duration_analysis["actual_duration_minutes"]
    - flight_duration_analysis["scheduled_duration_minutes"]
)

print("Total matched records:", len(flight_duration_analysis))
print(
    "Missing schedules:",
    flight_duration_analysis["scheduled_duration_minutes"].isna().sum()
)

display(
    flight_duration_analysis[
        [
            "bag_id",
            "flight_number",
            "source",
            "destination",
            "actual_duration_minutes",
            "scheduled_duration_minutes",
            "duration_difference_minutes"
        ]
    ].head(10)
)

Total matched records: 1500
Missing schedules: 0


,bag_id,flight_number,source,destination,actual_duration_minutes,scheduled_duration_minutes,duration_difference_minutes
0,BAG00001,6E5268,DEL,LHR,225.0,180.0,45.0
1,BAG00002,EK7887,DXB,JFK,405.0,360.0,45.0
2,BAG00003,LH2805,DXB,JFK,405.0,360.0,45.0
3,BAG00004,LH4110,MAA,DEL,225.0,180.0,45.0
4,BAG00005,6E811,MAA,SIN,225.0,180.0,45.0
5,BAG00006,EK9395,BOM,DXB,225.0,180.0,45.0
6,BAG00007,LH3358,FRA,LHR,405.0,360.0,45.0
7,BAG00008,EK5413,MAA,DEL,225.0,180.0,45.0
8,BAG00009,6E1233,BOM,DXB,225.0,180.0,45.0
9,BAG00010,QR3532,SIN,LHR,405.0,360.0,45.0


In [25]:
# Analyze actual versus scheduled duration differences

print("Duration difference distribution:")
display(
    flight_duration_analysis[
        "duration_difference_minutes"
    ].value_counts().sort_index()
)

print("\nDifference statistics:")
display(
    flight_duration_analysis[
        "duration_difference_minutes"
    ].describe()
)

print(
    "\nRecords with zero difference:",
    (flight_duration_analysis["duration_difference_minutes"] == 0).sum()
)

print(
    "Records with positive difference:",
    (flight_duration_analysis["duration_difference_minutes"] > 0).sum()
)

print(
    "Records with negative difference:",
    (flight_duration_analysis["duration_difference_minutes"] < 0).sum()
)

Duration difference distribution:


duration_difference_minutes
45.0    1500
Name: count, dtype: int64


Difference statistics:


count    1500.0
mean       45.0
std         0.0
min        45.0
25%        45.0
50%        45.0
75%        45.0
max        45.0
Name: duration_difference_minutes, dtype: float64


Records with zero difference: 0
Records with positive difference: 1500
Records with negative difference: 0


In [26]:
# Save actual versus scheduled flight duration analysis

flight_duration_output = (
    processed_path / "phase11_flight_duration_analysis.csv"
)

flight_duration_analysis.to_csv(
    flight_duration_output,
    index=False
)

print("Flight duration analysis saved!")
print("File:", flight_duration_output)
print("Total records:", len(flight_duration_analysis))
print(
    "Unique duration differences:",
    flight_duration_analysis["duration_difference_minutes"].nunique()
)

Flight duration analysis saved!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\phase11_flight_duration_analysis.csv
Total records: 1500
Unique duration differences: 1


In [27]:
# Prepare arrival and unloading events

arrived_events = baggage_scan_event[
    baggage_scan_event["event_type"] == "ARRIVED"
][["bag_id", "airport_code", "event_time"]].rename(
    columns={"event_time": "arrived_time"}
)

unloaded_events = baggage_scan_event[
    baggage_scan_event["event_type"] == "UNLOADED"
][["bag_id", "airport_code", "event_time"]].rename(
    columns={"event_time": "unloaded_time"}
)

arrival_unload = arrived_events.merge(
    unloaded_events,
    on=["bag_id", "airport_code"],
    how="inner",
    validate="one_to_one"
)

arrival_unload["arrival_to_unload_minutes"] = (
    arrival_unload["unloaded_time"] -
    arrival_unload["arrived_time"]
).dt.total_seconds() / 60

print("Matched arrival-unload records:", len(arrival_unload))

display(arrival_unload.head(10))

Matched arrival-unload records: 1500


,bag_id,airport_code,arrived_time,unloaded_time,arrival_to_unload_minutes
0,BAG00001,LHR,2026-11-01 18:00:00+00:00,2026-11-01 18:15:00+00:00,15.0
1,BAG00002,JFK,2026-11-02 12:00:00+00:00,2026-11-02 12:15:00+00:00,15.0
2,BAG00003,JFK,2026-11-03 12:00:00+00:00,2026-11-03 12:15:00+00:00,15.0
3,BAG00004,DEL,2026-11-04 09:00:00+00:00,2026-11-04 09:15:00+00:00,15.0
4,BAG00005,SIN,2026-11-05 09:00:00+00:00,2026-11-05 09:15:00+00:00,15.0
5,BAG00006,DXB,2026-11-06 18:00:00+00:00,2026-11-06 18:15:00+00:00,15.0
6,BAG00007,LHR,2026-11-07 12:00:00+00:00,2026-11-07 12:15:00+00:00,15.0
7,BAG00008,DEL,2026-11-08 09:00:00+00:00,2026-11-08 09:15:00+00:00,15.0
8,BAG00009,DXB,2026-11-09 18:00:00+00:00,2026-11-09 18:15:00+00:00,15.0
9,BAG00010,LHR,2026-11-10 21:00:00+00:00,2026-11-10 21:15:00+00:00,15.0


In [29]:
# Select unloading events at the connecting airport only

first_flight_unloaded = unloaded_events.merge(
    first_flight,
    left_on="bag_id",
    right_on="bag_id",
    how="inner",
    validate="many_to_one"
)

first_flight_unloaded = first_flight_unloaded[
    first_flight_unloaded["airport_code"]
    == first_flight_unloaded["connecting_airport"]
].copy()

first_flight_unloaded = first_flight_unloaded[
    ["bag_id", "airport_code", "unloaded_time"]
]

print("First-flight unloading records:", len(first_flight_unloaded))
print("Unique baggage:", first_flight_unloaded["bag_id"].nunique())

display(first_flight_unloaded.head(10))

First-flight unloading records: 1000
Unique baggage: 1000


,bag_id,airport_code,unloaded_time
0,BAG00001,LHR,2026-11-01 18:15:00+00:00
1,BAG00002,JFK,2026-11-02 12:15:00+00:00
2,BAG00003,JFK,2026-11-03 12:15:00+00:00
3,BAG00004,DEL,2026-11-04 09:15:00+00:00
4,BAG00005,SIN,2026-11-05 09:15:00+00:00
5,BAG00006,DXB,2026-11-06 18:15:00+00:00
6,BAG00007,LHR,2026-11-07 12:15:00+00:00
7,BAG00008,DEL,2026-11-08 09:15:00+00:00
8,BAG00009,DXB,2026-11-09 18:15:00+00:00
9,BAG00010,LHR,2026-11-10 21:15:00+00:00


In [30]:
# Keep only connecting baggage unloading records

first_flight_unloaded = first_flight_unloaded[
    first_flight_unloaded["bag_id"].isin(connecting_bag_ids)
].copy()

print("Connecting baggage unloading records:", len(first_flight_unloaded))
print("Unique baggage:", first_flight_unloaded["bag_id"].nunique())

display(first_flight_unloaded.head(10))

Connecting baggage unloading records: 500
Unique baggage: 500


,bag_id,airport_code,unloaded_time
500,BAG00501,DXB,2026-11-01 18:15:00+00:00
502,BAG00502,DXB,2026-11-02 09:15:00+00:00
504,BAG00503,DOH,2026-11-03 09:15:00+00:00
506,BAG00504,DOH,2026-11-04 09:15:00+00:00
508,BAG00505,DOH,2026-11-05 09:15:00+00:00
510,BAG00506,DOH,2026-11-06 18:15:00+00:00
512,BAG00507,DOH,2026-11-07 09:15:00+00:00
514,BAG00508,DOH,2026-11-08 09:15:00+00:00
516,BAG00509,DOH,2026-11-09 18:15:00+00:00
518,BAG00510,SIN,2026-11-10 18:15:00+00:00


In [31]:
# Calculate unloading-to-transfer duration

unloaded_transfer = first_flight_unloaded.merge(
    transfer_events[
        ["bag_id", "transfer_airport", "transfer_time"]
    ],
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

unloaded_transfer["airport_matches"] = (
    unloaded_transfer["airport_code"]
    == unloaded_transfer["transfer_airport"]
)

unloaded_transfer["unload_to_transfer_minutes"] = (
    unloaded_transfer["transfer_time"]
    - unloaded_transfer["unloaded_time"]
).dt.total_seconds() / 60

print("Matched connecting baggage:", len(unloaded_transfer))
print(
    "Airport mismatches:",
    (~unloaded_transfer["airport_matches"]).sum()
)

display(unloaded_transfer.head(10))

Matched connecting baggage: 500
Airport mismatches: 0


,bag_id,airport_code,unloaded_time,transfer_airport,transfer_time,airport_matches,unload_to_transfer_minutes
0,BAG00501,DXB,2026-11-01 18:15:00+00:00,DXB,2026-11-01 18:45:00+00:00,True,30.0
1,BAG00502,DXB,2026-11-02 09:15:00+00:00,DXB,2026-11-02 09:45:00+00:00,True,30.0
2,BAG00503,DOH,2026-11-03 09:15:00+00:00,DOH,2026-11-03 09:45:00+00:00,True,30.0
3,BAG00504,DOH,2026-11-04 09:15:00+00:00,DOH,2026-11-04 09:45:00+00:00,True,30.0
4,BAG00505,DOH,2026-11-05 09:15:00+00:00,DOH,2026-11-05 09:45:00+00:00,True,30.0
5,BAG00506,DOH,2026-11-06 18:15:00+00:00,DOH,2026-11-06 18:45:00+00:00,True,30.0
6,BAG00507,DOH,2026-11-07 09:15:00+00:00,DOH,2026-11-07 09:45:00+00:00,True,30.0
7,BAG00508,DOH,2026-11-08 09:15:00+00:00,DOH,2026-11-08 09:45:00+00:00,True,30.0
8,BAG00509,DOH,2026-11-09 18:15:00+00:00,DOH,2026-11-09 18:45:00+00:00,True,30.0
9,BAG00510,SIN,2026-11-10 18:15:00+00:00,SIN,2026-11-10 18:45:00+00:00,True,30.0


In [32]:
# Calculate transfer-to-second-flight loading duration

transfer_to_second_load = transfer_events.merge(
    second_loaded,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

transfer_to_second_load["transfer_to_load_minutes"] = (
    transfer_to_second_load["second_loaded_time"]
    - transfer_to_second_load["transfer_time"]
).dt.total_seconds() / 60

print("Matched connecting baggage:", len(transfer_to_second_load))

display(transfer_to_second_load.head(10))

Matched connecting baggage: 500


,bag_id,transfer_airport,transfer_time,second_loaded_airport,second_loaded_time,transfer_to_load_minutes
0,BAG00501,DXB,2026-11-01 18:45:00+00:00,DXB,2026-11-01 05:15:00+00:00,-810.0
1,BAG00502,DXB,2026-11-02 09:45:00+00:00,DXB,2026-11-02 14:15:00+00:00,270.0
2,BAG00503,DOH,2026-11-03 09:45:00+00:00,DOH,2026-11-03 14:15:00+00:00,270.0
3,BAG00504,DOH,2026-11-04 09:45:00+00:00,DOH,2026-11-04 14:15:00+00:00,270.0
4,BAG00505,DOH,2026-11-05 09:45:00+00:00,DOH,2026-11-05 14:15:00+00:00,270.0
5,BAG00506,DOH,2026-11-06 18:45:00+00:00,DOH,2026-11-06 05:15:00+00:00,-810.0
6,BAG00507,DOH,2026-11-07 09:45:00+00:00,DOH,2026-11-07 14:15:00+00:00,270.0
7,BAG00508,DOH,2026-11-08 09:45:00+00:00,DOH,2026-11-08 14:15:00+00:00,270.0
8,BAG00509,DOH,2026-11-09 18:45:00+00:00,DOH,2026-11-09 05:15:00+00:00,-810.0
9,BAG00510,SIN,2026-11-10 18:45:00+00:00,SIN,2026-11-10 05:15:00+00:00,-810.0


In [33]:
# Combine connecting baggage dwell-time analysis

dwell_time_report = (
    unloaded_transfer[
        [
            "bag_id",
            "airport_code",
            "unloaded_time",
            "transfer_time",
            "unload_to_transfer_minutes"
        ]
    ]
    .merge(
        transfer_to_second_load[
            [
                "bag_id",
                "second_loaded_time",
                "transfer_to_load_minutes"
            ]
        ],
        on="bag_id",
        how="inner",
        validate="one_to_one"
    )
)

dwell_time_report["handoff_chronology_valid"] = (
    dwell_time_report["transfer_to_load_minutes"] >= 0
)

dwell_output_path = (
    processed_path / "phase11_dwell_time_analysis.csv"
)

dwell_time_report.to_csv(dwell_output_path, index=False)

print("Dwell-time report saved!")
print("Records:", len(dwell_time_report))
print("\nChronology distribution:")
display(
    dwell_time_report["handoff_chronology_valid"].value_counts()
)

Dwell-time report saved!
Records: 500

Chronology distribution:


handoff_chronology_valid
True     315
False    185
Name: count, dtype: int64

In [34]:
# Load original Phase 5 investigation results

phase5_path = (
    processed_path / "final_baggage_misroute_investigation.csv"
)

phase5_investigation = pd.read_csv(phase5_path)

print("Dataset shape:", phase5_investigation.shape)

print("\nColumn names:")
print(phase5_investigation.columns.tolist())

print("\nSample records:")
display(phase5_investigation.head())

Dataset shape: (1000, 12)

Column names:
['bag_id', 'unexpected_airport_detected', 'incorrect_flight_detected', 'incorrect_airport_flight_detected', 'segment_event_investigation_required', 'connection_gap_minutes', 'impossible_connection', 'handoff_gap_minutes', 'loaded_before_transfer', 'journey_type', 'event_sequence_matches', 'investigation_status']

Sample records:


,bag_id,unexpected_airport_detected,incorrect_flight_detected,incorrect_airport_flight_detected,segment_event_investigation_required,connection_gap_minutes,impossible_connection,handoff_gap_minutes,loaded_before_transfer,journey_type,event_sequence_matches,investigation_status
0,BAG00001,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
1,BAG00002,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
2,BAG00003,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
3,BAG00004,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
4,BAG00005,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED


In [35]:
# Validate Phase 5 baggage identifiers

print("Total records:", len(phase5_investigation))

print(
    "Unique baggage IDs:",
    phase5_investigation["bag_id"].nunique()
)

print(
    "Duplicate baggage IDs:",
    phase5_investigation["bag_id"].duplicated().sum()
)

print(
    "Missing baggage IDs:",
    phase5_investigation["bag_id"].isna().sum()
)

print("\nOriginal investigation status distribution:")

display(
    phase5_investigation["investigation_status"].value_counts()
)

Total records: 1000
Unique baggage IDs: 1000
Duplicate baggage IDs: 0
Missing baggage IDs: 0

Original investigation status distribution:


investigation_status
NO_ISSUE_DETECTED       500
EVENT_PATTERN_REVIEW    315
TIMING_INCONSISTENCY    185
Name: count, dtype: int64

In [36]:
# Aggregate missing segment events at baggage level

missing_event_summary = (
    missing_events_df
    .groupby("bag_id")
    .agg(
        missing_segment_event_count=("missing_event", "count"),
        missing_segment_events=(
            "missing_event",
            lambda values: ", ".join(sorted(set(values)))
        )
    )
    .reset_index()
)

print("Baggage with missing segment events:",
      missing_event_summary["bag_id"].nunique())

print("Total missing segment events:",
      missing_event_summary["missing_segment_event_count"].sum())

display(missing_event_summary.head(10))

Baggage with missing segment events: 500
Total missing segment events: 500


,bag_id,missing_segment_event_count,missing_segment_events
0,BAG00501,1,SORTED
1,BAG00502,1,SORTED
2,BAG00503,1,SORTED
3,BAG00504,1,SORTED
4,BAG00505,1,SORTED
5,BAG00506,1,SORTED
6,BAG00507,1,SORTED
7,BAG00508,1,SORTED
8,BAG00509,1,SORTED
9,BAG00510,1,SORTED


In [37]:
# Prepare baggage-level handoff summary

handoff_summary_final = handoff_report[
    [
        "bag_id",
        "unload_to_transfer_minutes",
        "transfer_to_load_minutes",
        "airport_consistent",
        "handoff_chronology_valid",
        "handoff_issue"
    ]
].copy()

print("Handoff records:", len(handoff_summary_final))
print(
    "Unique baggage IDs:",
    handoff_summary_final["bag_id"].nunique()
)

print("\nHandoff issue distribution:")
display(handoff_summary_final["handoff_issue"].value_counts())

display(handoff_summary_final.head(10))

Handoff records: 500
Unique baggage IDs: 500

Handoff issue distribution:


handoff_issue
NO_TIMING_ISSUE                        315
TRANSFER_LOADING_TIME_INCONSISTENCY    185
Name: count, dtype: int64

,bag_id,unload_to_transfer_minutes,transfer_to_load_minutes,airport_consistent,handoff_chronology_valid,handoff_issue
0,BAG00501,30.0,-810.0,True,False,TRANSFER_LOADING_TIME_INCONSISTENCY
1,BAG00502,30.0,270.0,True,True,NO_TIMING_ISSUE
2,BAG00503,30.0,270.0,True,True,NO_TIMING_ISSUE
3,BAG00504,30.0,270.0,True,True,NO_TIMING_ISSUE
4,BAG00505,30.0,270.0,True,True,NO_TIMING_ISSUE
5,BAG00506,30.0,-810.0,True,False,TRANSFER_LOADING_TIME_INCONSISTENCY
6,BAG00507,30.0,270.0,True,True,NO_TIMING_ISSUE
7,BAG00508,30.0,270.0,True,True,NO_TIMING_ISSUE
8,BAG00509,30.0,-810.0,True,False,TRANSFER_LOADING_TIME_INCONSISTENCY
9,BAG00510,30.0,-810.0,True,False,TRANSFER_LOADING_TIME_INCONSISTENCY


In [38]:
# Prepare flight duration summary at baggage level

flight_duration_summary = flight_duration_analysis[
    [
        "bag_id",
        "actual_duration_minutes",
        "scheduled_duration_minutes",
        "duration_difference_minutes"
    ]
].copy()

print("Flight duration records:", len(flight_duration_summary))

print(
    "Unique baggage IDs:",
    flight_duration_summary["bag_id"].nunique()
)

print("\nDuration difference distribution:")
display(
    flight_duration_summary[
        "duration_difference_minutes"
    ].value_counts()
)

display(flight_duration_summary.head(10))

Flight duration records: 1500
Unique baggage IDs: 1000

Duration difference distribution:


duration_difference_minutes
45.0    1500
Name: count, dtype: int64

,bag_id,actual_duration_minutes,scheduled_duration_minutes,duration_difference_minutes
0,BAG00001,225.0,180.0,45.0
1,BAG00002,405.0,360.0,45.0
2,BAG00003,405.0,360.0,45.0
3,BAG00004,225.0,180.0,45.0
4,BAG00005,225.0,180.0,45.0
5,BAG00006,225.0,180.0,45.0
6,BAG00007,405.0,360.0,45.0
7,BAG00008,225.0,180.0,45.0
8,BAG00009,225.0,180.0,45.0
9,BAG00010,405.0,360.0,45.0


In [39]:
# Aggregate flight duration records at baggage level

flight_duration_bag_summary = (
    flight_duration_summary
    .groupby("bag_id")
    .agg(
        total_flight_segments=("flight_number", "count")
        if "flight_number" in flight_duration_summary.columns
        else ("actual_duration_minutes", "count"),
        total_actual_duration_minutes=("actual_duration_minutes", "sum"),
        total_scheduled_duration_minutes=("scheduled_duration_minutes", "sum"),
        total_duration_difference_minutes=("duration_difference_minutes", "sum")
    )
    .reset_index()
)

print("Total baggage records:", len(flight_duration_bag_summary))
print(
    "Unique baggage IDs:",
    flight_duration_bag_summary["bag_id"].nunique()
)

print("\nFlight segment distribution:")
display(
    flight_duration_bag_summary["total_flight_segments"].value_counts()
)

display(flight_duration_bag_summary.head(10))

Total baggage records: 1000
Unique baggage IDs: 1000

Flight segment distribution:


total_flight_segments
1    500
2    500
Name: count, dtype: int64

,bag_id,total_flight_segments,total_actual_duration_minutes,total_scheduled_duration_minutes,total_duration_difference_minutes
0,BAG00001,1,225.0,180.0,45.0
1,BAG00002,1,405.0,360.0,45.0
2,BAG00003,1,405.0,360.0,45.0
3,BAG00004,1,225.0,180.0,45.0
4,BAG00005,1,225.0,180.0,45.0
5,BAG00006,1,225.0,180.0,45.0
6,BAG00007,1,405.0,360.0,45.0
7,BAG00008,1,225.0,180.0,45.0
8,BAG00009,1,225.0,180.0,45.0
9,BAG00010,1,405.0,360.0,45.0


In [40]:
# Validate flight duration aggregation by segment count

duration_by_journey = (
    flight_duration_bag_summary
    .groupby("total_flight_segments")
    .agg(
        baggage_count=("bag_id", "count"),
        average_actual_duration=(
            "total_actual_duration_minutes", "mean"
        ),
        average_scheduled_duration=(
            "total_scheduled_duration_minutes", "mean"
        ),
        average_duration_difference=(
            "total_duration_difference_minutes", "mean"
        )
    )
    .reset_index()
)

display(duration_by_journey.round(2))

,total_flight_segments,baggage_count,average_actual_duration,average_scheduled_duration,average_duration_difference
0,1,500,303.84,258.84,45.0
1,2,500,602.28,512.28,90.0


In [41]:
# Start with the original Phase 5 investigation dataset

advanced_investigation = phase5_investigation.copy()

# Merge missing segment-event summary
advanced_investigation = advanced_investigation.merge(
    missing_event_summary,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge handoff summary
advanced_investigation = advanced_investigation.merge(
    handoff_summary_final,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge baggage-level flight duration summary
advanced_investigation = advanced_investigation.merge(
    flight_duration_bag_summary,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

print("Consolidated dataset shape:", advanced_investigation.shape)
print("Unique baggage IDs:", advanced_investigation["bag_id"].nunique())

print("\nMissing values by column:")
display(advanced_investigation.isna().sum())

display(advanced_investigation.head())

Consolidated dataset shape: (1000, 23)
Unique baggage IDs: 1000

Missing values by column:


bag_id                                    0
unexpected_airport_detected               0
incorrect_flight_detected                 0
incorrect_airport_flight_detected         0
segment_event_investigation_required      0
connection_gap_minutes                  500
impossible_connection                     0
handoff_gap_minutes                     500
loaded_before_transfer                    0
journey_type                              0
event_sequence_matches                    0
investigation_status                      0
missing_segment_event_count             500
missing_segment_events                  500
unload_to_transfer_minutes              500
transfer_to_load_minutes                500
airport_consistent                      500
handoff_chronology_valid                500
handoff_issue                           500
total_flight_segments                     0
total_actual_duration_minutes             0
total_scheduled_duration_minutes          0
total_duration_difference_minute

,bag_id,unexpected_airport_detected,incorrect_flight_detected,incorrect_airport_flight_detected,segment_event_investigation_required,connection_gap_minutes,impossible_connection,handoff_gap_minutes,loaded_before_transfer,journey_type,...,missing_segment_events,unload_to_transfer_minutes,transfer_to_load_minutes,airport_consistent,handoff_chronology_valid,handoff_issue,total_flight_segments,total_actual_duration_minutes,total_scheduled_duration_minutes,total_duration_difference_minutes
0,BAG00001,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,NaN,NaN,NaN,1,225.0,180.0,45.0
1,BAG00002,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,NaN,NaN,NaN,1,405.0,360.0,45.0
2,BAG00003,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,NaN,NaN,NaN,1,405.0,360.0,45.0
3,BAG00004,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,NaN,NaN,NaN,1,225.0,180.0,45.0
4,BAG00005,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,NaN,NaN,NaN,1,225.0,180.0,45.0


In [42]:
# Fill missing-event summary for baggage with no detected missing events

advanced_investigation["missing_segment_event_count"] = (
    advanced_investigation["missing_segment_event_count"].fillna(0).astype(int)
)

advanced_investigation["missing_segment_events"] = (
    advanced_investigation["missing_segment_events"].fillna("NONE")
)

print("Missing-event count distribution:")
display(
    advanced_investigation["missing_segment_event_count"].value_counts()
)

print("\nMissing-event description distribution:")
display(
    advanced_investigation["missing_segment_events"].value_counts()
)

print("\nRemaining missing values:")
display(advanced_investigation.isna().sum())

Missing-event count distribution:


missing_segment_event_count
0    500
1    500
Name: count, dtype: int64


Missing-event description distribution:


missing_segment_events
NONE      500
SORTED    500
Name: count, dtype: int64


Remaining missing values:


bag_id                                    0
unexpected_airport_detected               0
incorrect_flight_detected                 0
incorrect_airport_flight_detected         0
segment_event_investigation_required      0
connection_gap_minutes                  500
impossible_connection                     0
handoff_gap_minutes                     500
loaded_before_transfer                    0
journey_type                              0
event_sequence_matches                    0
investigation_status                      0
missing_segment_event_count               0
missing_segment_events                    0
unload_to_transfer_minutes              500
transfer_to_load_minutes                500
airport_consistent                      500
handoff_chronology_valid                500
handoff_issue                           500
total_flight_segments                     0
total_actual_duration_minutes             0
total_scheduled_duration_minutes          0
total_duration_difference_minute

In [43]:
# Aggregate arrival-to-unloading duration at baggage level

arrival_unload_summary = (
    arrival_unload
    .groupby("bag_id")
    .agg(
        total_arrival_unload_events=("arrival_to_unload_minutes", "count"),
        average_arrival_to_unload_minutes=(
            "arrival_to_unload_minutes", "mean"
        ),
        total_arrival_to_unload_minutes=(
            "arrival_to_unload_minutes", "sum"
        )
    )
    .reset_index()
)

print("Baggage records:", len(arrival_unload_summary))
print(
    "Unique baggage IDs:",
    arrival_unload_summary["bag_id"].nunique()
)

display(arrival_unload_summary.head(10))

Baggage records: 1000
Unique baggage IDs: 1000


,bag_id,total_arrival_unload_events,average_arrival_to_unload_minutes,total_arrival_to_unload_minutes
0,BAG00001,1,15.0,15.0
1,BAG00002,1,15.0,15.0
2,BAG00003,1,15.0,15.0
3,BAG00004,1,15.0,15.0
4,BAG00005,1,15.0,15.0
5,BAG00006,1,15.0,15.0
6,BAG00007,1,15.0,15.0
7,BAG00008,1,15.0,15.0
8,BAG00009,1,15.0,15.0
9,BAG00010,1,15.0,15.0


In [44]:
# Merge arrival-to-unloading summary

advanced_investigation = advanced_investigation.merge(
    arrival_unload_summary,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

print("Updated dataset shape:", advanced_investigation.shape)

print(
    "Unique baggage IDs:",
    advanced_investigation["bag_id"].nunique()
)

print("\nArrival-to-unloading event distribution:")

display(
    advanced_investigation[
        "total_arrival_unload_events"
    ].value_counts(dropna=False)
)

display(advanced_investigation.head())

Updated dataset shape: (1000, 26)
Unique baggage IDs: 1000

Arrival-to-unloading event distribution:


total_arrival_unload_events
1    500
2    500
Name: count, dtype: int64

,bag_id,unexpected_airport_detected,incorrect_flight_detected,incorrect_airport_flight_detected,segment_event_investigation_required,connection_gap_minutes,impossible_connection,handoff_gap_minutes,loaded_before_transfer,journey_type,...,airport_consistent,handoff_chronology_valid,handoff_issue,total_flight_segments,total_actual_duration_minutes,total_scheduled_duration_minutes,total_duration_difference_minutes,total_arrival_unload_events,average_arrival_to_unload_minutes,total_arrival_to_unload_minutes
0,BAG00001,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,1,225.0,180.0,45.0,1,15.0,15.0
1,BAG00002,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,1,405.0,360.0,45.0,1,15.0,15.0
2,BAG00003,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,1,405.0,360.0,45.0,1,15.0,15.0
3,BAG00004,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,1,225.0,180.0,45.0,1,15.0,15.0
4,BAG00005,False,False,False,False,NaN,False,NaN,False,DIRECT,...,NaN,NaN,NaN,1,225.0,180.0,45.0,1,15.0,15.0


In [45]:
# Create advanced detection flags

advanced_investigation["missing_segment_event_detected"] = (
    advanced_investigation["missing_segment_event_count"] > 0
)

advanced_investigation["handoff_timing_inconsistency_detected"] = (
    advanced_investigation["handoff_chronology_valid"] == False
).fillna(False)

advanced_investigation["airport_consistency_issue_detected"] = (
    advanced_investigation["airport_consistent"] == False
).fillna(False)

# Arrival-to-unloading duration is consistently 15 minutes.
# We do not classify the uniform dataset pattern as an anomaly.
advanced_investigation["event_duration_pattern_detected"] = False

print("Advanced detection flag counts:")

flag_columns = [
    "missing_segment_event_detected",
    "handoff_timing_inconsistency_detected",
    "airport_consistency_issue_detected",
    "event_duration_pattern_detected"
]

for column in flag_columns:
    print(f"\n{column}")
    display(advanced_investigation[column].value_counts())

Advanced detection flag counts:

missing_segment_event_detected


missing_segment_event_detected
False    500
True     500
Name: count, dtype: int64


handoff_timing_inconsistency_detected


handoff_timing_inconsistency_detected
False    815
True     185
Name: count, dtype: int64


airport_consistency_issue_detected


airport_consistency_issue_detected
False    1000
Name: count, dtype: int64


event_duration_pattern_detected


event_duration_pattern_detected
False    1000
Name: count, dtype: int64

In [46]:
# Classify combined advanced detection findings

missing_flag = advanced_investigation[
    "missing_segment_event_detected"
]

handoff_flag = advanced_investigation[
    "handoff_timing_inconsistency_detected"
]

advanced_investigation["advanced_detection_status"] = np.select(
    [
        missing_flag & handoff_flag,
        missing_flag & ~handoff_flag,
        ~missing_flag & handoff_flag
    ],
    [
        "MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE",
        "MISSING_EVENT_ONLY",
        "HANDOFF_TIMING_ISSUE_ONLY"
    ],
    default="NO_ADVANCED_ISSUE_DETECTED"
)

advanced_investigation["advanced_issue_detected"] = (
    missing_flag | handoff_flag
)

print("Advanced investigation status distribution:")

display(
    advanced_investigation[
        "advanced_detection_status"
    ].value_counts()
)

print("\nUnique bags with advanced findings:")
print(
    advanced_investigation["advanced_issue_detected"].sum()
)

Advanced investigation status distribution:


advanced_detection_status
NO_ADVANCED_ISSUE_DETECTED                500
MISSING_EVENT_ONLY                        315
MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE    185
Name: count, dtype: int64


Unique bags with advanced findings:
500


In [47]:
# Assign advanced investigation severity

advanced_investigation["advanced_severity"] = np.select(
    [
        (
            advanced_investigation["missing_segment_event_detected"]
            & advanced_investigation["handoff_timing_inconsistency_detected"]
        ),
        advanced_investigation["missing_segment_event_detected"]
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

print("Advanced severity distribution:")

display(
    advanced_investigation["advanced_severity"].value_counts()
)

print("\nSeverity by journey type:")

display(
    pd.crosstab(
        advanced_investigation["journey_type"],
        advanced_investigation["advanced_severity"]
    )
)

Advanced severity distribution:


advanced_severity
LOW       500
MEDIUM    315
HIGH      185
Name: count, dtype: int64


Severity by journey type:


advanced_severity,HIGH,LOW,MEDIUM
journey_type,,,
CONNECTING,185,0,315
DIRECT,0,500,0


In [48]:
# Save final Phase 11 advanced investigation dataset

final_advanced_path = (
    processed_path / "final_phase11_advanced_investigation.csv"
)

advanced_investigation.to_csv(
    final_advanced_path,
    index=False
)

print("Phase 11 final dataset saved!")
print("File:", final_advanced_path)
print("Total records:", len(advanced_investigation))
print(
    "Unique baggage IDs:",
    advanced_investigation["bag_id"].nunique()
)

print("\nFinal severity distribution:")
display(
    advanced_investigation["advanced_severity"].value_counts()
)

Phase 11 final dataset saved!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\final_phase11_advanced_investigation.csv
Total records: 1000
Unique baggage IDs: 1000

Final severity distribution:


advanced_severity
LOW       500
MEDIUM    315
HIGH      185
Name: count, dtype: int64

In [49]:
# Final Phase 11 validation

original_phase5 = phase5_investigation.set_index("bag_id")
final_phase11 = advanced_investigation.set_index("bag_id")

print("Total records:", len(final_phase11))
print("Unique baggage IDs:", final_phase11.index.nunique())
print("Duplicate baggage IDs:", final_phase11.index.duplicated().sum())

print("\nJourney type distribution:")
display(final_phase11["journey_type"].value_counts())

print("\nOriginal vs final investigation status:")

print(
    "Statuses unchanged:",
    original_phase5["investigation_status"].sort_index().equals(
        final_phase11["investigation_status"].sort_index()
    )
)

print("\nFinal severity distribution:")
display(final_phase11["advanced_severity"].value_counts())

print("\nAdvanced findings:")
print(
    "Unique bags with advanced findings:",
    final_phase11["advanced_issue_detected"].sum()
)

Total records: 1000
Unique baggage IDs: 1000
Duplicate baggage IDs: 0

Journey type distribution:


journey_type
DIRECT        500
CONNECTING    500
Name: count, dtype: int64


Original vs final investigation status:
Statuses unchanged: True

Final severity distribution:


advanced_severity
LOW       500
MEDIUM    315
HIGH      185
Name: count, dtype: int64


Advanced findings:
Unique bags with advanced findings: 500
